# 1 — Alignment and event QA

Checks that every trial lands on the same axes and that the detected ground
contacts are physiologically possible, *before* any modelling.

Vertical comes from the raw Xsens Z axis (the suit measures gravity, so vertical
is observed rather than inferred). Heading is estimated over the window being
analysed rather than over the whole 60 m, so lane drift and the curved path out
of the blocks cannot tilt it. See `sprint/frame.py`.

Set `SPRINT_C3D_DIR` to the folder holding the trials before running.

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sprint import config as C, events, features, figures, io, model, skeleton

## Per-trial QA

`v = step length x step frequency` has to close, and contact time has to be physiologically possible. A trial that fails either is a detection problem, not a finding.

In [ ]:
rows = []
for path in sorted(C.C3D_DIR.glob("*.c3d")):
    pid = path.stem.split("-")[0].strip()
    if pid in C.EXCLUDED_PIDS:
        continue
    markers, labels, fs = io.load_c3d(path)
    idx = io.role_index(labels)
    vel, _ = events.velocity(markers, idx, fs)
    df = events.step_table(markers, idx, fs)
    rows.append(dict(participant_id=pid, **events.qa(df, float(vel.max()))))

qa = pd.DataFrame(rows)
qa

In [ ]:
for col in ("gct_in_range", "duty_in_range", "v_matches_sl_x_sf"):
    bad = qa.loc[~qa[col].astype(bool), "participant_id"].tolist()
    print(f"{col}: {len(qa) - len(bad)}/{len(qa)} pass", f"failing: {bad}" if bad else "")

## Skeleton grid

One mid-contact pose per athlete, all in the same local frame. Anything that looks rotated or mis-scaled here is an alignment failure.

In [ ]:
poses, pids = [], []
for path in sorted(C.C3D_DIR.glob("*.c3d")):
    pid = path.stem.split("-")[0].strip()
    if pid in C.EXCLUDED_PIDS:
        continue
    markers, labels, fs = io.load_c3d(path)
    idx = io.role_index(labels)
    _, _, steps, rep = features.build(markers, idx, fs, "topspeed")
    tensor = features.step_tensor(markers, idx, steps, None, rep["stature"])
    poses.append(np.nanmean(tensor, axis=0)[C.N_CONTACT // 2]); pids.append(pid)

ncol = 6
fig, axes = plt.subplots(-(-len(poses) // ncol), ncol, figsize=(2.1 * ncol, 2.6 * -(-len(poses) // ncol)))
for ax, pose, pid in zip(axes.ravel(), poses, pids):
    skeleton.draw(ax, pose); ax.set_title(pid, fontsize=8); ax.set_aspect("equal"); ax.axis("off")
for ax in axes.ravel()[len(poses):]:
    ax.axis("off")
fig.tight_layout()